In [1]:
import os, sys, time
sys.path.append(os.path.abspath(os.path.join(os.curdir, '..')))
from src.utils import *
from src.panda_program import PandaMugProgram
from src.generic_program import ProgramOptions
from pydrake.all import (
    StartMeshcat,
    Quaternion,
    RigidTransform,
    RotationMatrix,
    MinimumDistanceLowerBoundConstraint,
    SceneGraphInspector,
)



ikflow/config.py | Using device: 'cuda:0'


In [2]:
yaml_file = os.path.join(RepoDir(), "models/panda/panda_finray_collision.yaml")
meshcat = StartMeshcat()
diagram = BuildEnv(meshcat=meshcat, directives_file = yaml_file)
options = ProgramOptions(
    vars_file="panda_mug_joints.txt",
    ik_constraint_tol=(1e-3, 0),
)
program = PandaMugProgram(diagram)


program.plant.num_positions()
program.create_prog()



INFO:drake:Meshcat listening for connections at http://localhost:7000


WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/panda_arm_hand_formatted_link_filepaths_absolute.urdf
joint mimic: no multiplier, using default value of 1 
joint mimic: no offset, using default value of 0 
URDFParser: Link size: 17
URDFParser: Joint size: 12
Geometry: Loading 12 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link0.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link3.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link4.dae into Group
Geometry: Loading 3 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link5.dae into Group
Geometry: Loading 17 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link6.dae into Group
Geometry: Loa

In [3]:
num_tests = 100
start = time.time()
i = 0
q = np.zeros(7)
qs = np.zeros((num_tests, 7))
targets = np.zeros((num_tests, 7))
while i < num_tests:
    q = np.random.uniform(program.plant.GetPositionLowerLimits(), program.plant.GetPositionUpperLimits())
    program.plant.SetPositions(program.plant_context, q)
    if program.collision_free_constraint_eval.Eval(q) < 1:
        pose = program.frame.CalcPoseInWorld(program.plant_context)
        targets[i] = np.array([*pose.translation(), *pose.rotation().ToQuaternion().wxyz()])
        qs[i] = q
        i += 1
print("Generated {} collision-free targets in {:.2f} seconds".format(num_tests, time.time() - start))


def GenerateDiagramWithMug(q, yaml_file = yaml_file):
    program.SetPositions(q)
    target = program.frame.CalcPoseInWorld(program.plant_context)
    translation = target.translation()
    rotation = target.rotation().ToRollPitchYaw().vector() * 180 / np.pi

    mug_str = f"""\n  - add_model:\n      name: mug\n      # file: package://drake/examples/manipulation_station/models/shelves.sdf\n      file: package://combining_kinematics/models/mug/mug_simple_red.urdf\n  - add_weld:\n      parent: world\n      child: mug::mug_body_link\n      X_PC:\n        translation: [{translation[0]}, {translation[1]}, {translation[2]}]\n        rotation: !Rpy {{ deg: [{rotation[0]}, {rotation[1]}, {rotation[2]}] }}
    """
    original_size = os.path.getsize(yaml_file)
    with open(yaml_file, "a+") as f:
        f.write(mug_str)
    meshcat_with_mug = StartMeshcat()
    diagram_with_mug = BuildEnv(meshcat=meshcat_with_mug, directives_file = yaml_file)
    with open(yaml_file, "r+") as f:
        f.truncate(original_size)
    return diagram_with_mug, meshcat_with_mug, Mug(target)




Generated 100 collision-free targets in 0.15 seconds


In [4]:
diagram_with_mug, meshcat_with_mug, mug = GenerateDiagramWithMug(qs[1])
program = PandaMugProgram(diagram_with_mug, options=options)
program.SetPositions(qs[1])
program.diagram.ForcedPublish(program.diagram_context)

INFO:drake:Meshcat listening for connections at http://localhost:7001


WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/panda_arm_hand_formatted_link_filepaths_absolute.urdf
joint mimic: no multiplier, using default value of 1 
joint mimic: no offset, using default value of 0 
URDFParser: Link size: 17
URDFParser: Joint size: 12
Geometry: Loading 12 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link0.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link3.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link4.dae into Group
Geometry: Loading 3 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link5.dae into Group
Geometry: Loading 17 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link6.dae into Group
ManagedGeomet

In [5]:
with open(program.options.vars_file, "r+") as f:
    f.truncate(0)
program.create_prog(target_mug=mug)
result = program.Solve()
result.is_success()

True

In [6]:
q_sol = program.VarsToQ(result.get_x_val())
program.SetPositions(q_sol)
program.diagram.ForcedPublish(program.diagram_context)
print(program.collision_free_constraint_eval.Eval(q_sol))

[0.99450565]


In [7]:
with open(program.options.vars_file, "r") as f:
    lines = f.readlines()
    for line in lines:
        vars = [float(val) for val in line.strip().split(",")]
        # q = program.VarsToQ(vars)
        # program.SetPositions(q)
        # program.diagram.ForcedPublish(program.diagram_context)
        # time.sleep(0.1)
        DrawSphere(RigidTransform(vars[:3]), meshcat_with_mug)
        time.sleep(0.1)

In [8]:
diagram, meshcat2, mug = GenerateDiagramWithMug(qs[1], yaml_file = os.path.join(RepoDir(), "models/panda/panda_finray_collision_backup.yaml"))

INFO:drake:Meshcat listening for connections at http://localhost:7002


In [12]:
plant = diagram.GetSubsystemByName("plant")
diagram_context = diagram.CreateDefaultContext()
plant_context = plant.GetMyContextFromRoot(diagram_context)
diagram.ForcedPublish(diagram_context)

In [13]:
q_visual = np.zeros(14)
q_visual[:7] = qs[1]
plant.SetPositions(plant_context, q_visual)

meshcat2.SetProperty(f"/drake/illustration/panda2", "opacity", 0.3)
meshcat2.SetProperty(f"/drake/illustration/finray2", "opacity", 0.3)
meshcat2.SetProperty(f"/drake/illustration/mug", "opacity", 0.5)
meshcat2.SetProperty(f"/drake/illustration/shelves", "opacity", 0.2)
meshcat2.SetProperty(f"/drake/illustration/shelves2", "opacity", 0.2)
meshcat2.SetProperty(f"/drake/illustration/shelves3", "opacity", 0.2)
meshcat2.SetProperty(f"/drake/illustration/shelves4", "opacity", 0.2)
meshcat2.SetProperty(f"/drake/illustration/binF", "opacity", 0.2)

diagram.ForcedPublish(diagram_context)


In [20]:
with open(program.options.vars_file, "r") as f:
    lines = f.readlines()
    for line in lines:
        vars = [float(val) for val in line.strip().split(",")]
        q_visual[7:] = program.VarsToQ(vars, add_correction = False)
        q_visual[:7] = q_visual[7:] + vars[-7:]
        plant.SetPositions(plant_context, q_visual)
        diagram.ForcedPublish(diagram_context)
        time.sleep(0.2)